In [ ]:
import argparse
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


import os
import numpy as np
import torch
from argparse import ArgumentParser
import tqdm 

from config import *
from helpers import * 
import articulate as art
from constants import MODULES
from utils.model_utils import load_model
from data import PoseDataset
from models import MobilePoserNet

import argparse
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset



In [ ]:
import os
import math
import numpy as np
import torch
torch.set_printoptions(sci_mode=False)
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import lightning as L
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch import seed_everything
from argparse import ArgumentParser
from pathlib import Path
from typing import List
from tqdm import tqdm 
import wandb

from constants import MODULES
from data import PoseDataModule
from utils.file_utils import (
    get_datestring, 
    make_dir, 
    get_dir_number, 
    get_best_checkpoint
)
from config import paths, train_hypers, finetune_hypers


## dataset

In [ ]:
# amass_datasets

In [ ]:
from data import PoseDataset
import torch

ds = PoseDataset(fold='train', finetune=None)   # loads processed_datasets
pairs = []   # list of (pose_seq, tran_seq) per window/sequence
for i in range(len(ds)):
    item = ds[i]
    # training dataset returns (imu, pose, joint, tran, vel, contact)
    imu, pose, joint, tran = item[:4]
    # pose: (T, 6 * num_pred_joints) , tran: (T, 3)
    pairs.append((pose, tran))



In [ ]:
# # example: concatenate all sequences (if lengths match) or keep as list
all_poses = torch.cat([p for p, t in pairs], dim=0)
all_trans  = torch.cat([t for p, t in pairs], dim=0)

In [ ]:
all_poses.shape

In [ ]:
all_trans.shape

In [ ]:
device = "cuda"
x0 = torch.cat([all_poses.to(device), all_trans.to(device)], dim=-1)  # (B, 147)

In [ ]:

# # dataset_name = "dip"

# # if dataset_name not in datasets.test_datasets:
# #     raise ValueError(f"Test dataset: {dataset_name} not found.")
# # dataset = PoseDataset(fold="test", evaluate=dataset_name)

# # print(f"Starting evaluation: {dataset_name.capitalize()}")

# # device = "cuda"

# # # load data
# # xs, ys = zip(*[(imu.to(device), (pose.to(device), tran)) for imu, pose, joint, tran in dataset])

# # offline_errs, online_errs = [], []
# # tran_errors = {window_size: [] for window_size in list(range(1, 8))}


# # with torch.no_grad():
# #     for x, y in tqdm(list(zip(xs, ys))):
# #         pose_t, tran_t = y

# # x0 = torch.cat([pose_t.to(device), tran_t.to(device)], dim=-1)  # (B, 147)


# import torch, os
# import articulate as art
# from config import paths, amass, datasets

# bm = art.model.ParametricModel(paths.smpl_file)
# poses_list, trans_list = [], []

# data_folder = paths.processed_datasets  # or paths.processed_datasets / 'eval'
# for fname in os.listdir(data_folder):
#     if not fname.endswith('.pt'):
#         continue
#     data = torch.load(data_folder / fname)
#     for pose_mat, tran in zip(data['pose'], data['tran']):
#         # pose_mat shape typically (T, 24, 3, 3) or local rot; convert to global if needed:
#         pose_global, _ = bm.forward_kinematics(pose=pose_mat.view(-1, 216))
#         pose_rot = pose_global.view(-1, 24, 3, 3)          # (T,24,3,3)
#         r6 = art.math.rotation_matrix_to_r6d(pose_rot)      # (T,24,6)
#         # select predicted joints and flatten per time-step to (T, 6*num_pred_joints)
#         r6_sel = r6[:, amass.pred_joints_set, :].reshape(r6.shape[0], -1)
#         poses_list.append(r6_sel)
#         trans_list.append(tran)

# # keep as list of variable-length sequences or concatenate where appropriate

In [ ]:
def get_dataset(name):
    if name == "pose":
        return TensorDataset(x0)
    else:
        raise ValueError(f"Unknown dataset: {name}")

## model

In [ ]:


class Block(nn.Module):
    def __init__(self, size: int):
        super().__init__()
        self.ln = nn.LayerNorm(size)
        self.ff = nn.Linear(size, size)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor):
        return x + self.act(self.ff(self.ln(x)))

class SinusoidalEmbedding(nn.Module):
    def __init__(self, size: int, scale: float = 1.0):
        super().__init__()
        self.size = size
        self.scale = scale
        half_size = self.size // 2
        emb = torch.log(torch.Tensor([10000.0])) / (half_size - 1)
        emb = torch.exp(-emb * torch.arange(half_size))
        self.emb = nn.Parameter(emb, requires_grad=False)

    def forward(self, x: torch.Tensor):
        x = x * self.scale
        emb = x * self.emb.unsqueeze(0)
        emb = torch.cat((torch.sin(emb), torch.cos(emb)), dim=-1)
        return emb


class MLP(nn.Module):
    def __init__(self, data_dim=2, hidden_size=128, hidden_layers=3, emb_size=128):
        super().__init__()
        self.data_dim = data_dim

        self.time_emb = SinusoidalEmbedding(emb_size)

        self.concat_size = data_dim + emb_size
        self.data_size = data_dim

        layers = [nn.Linear(self.concat_size, hidden_size), nn.GELU()]
        for _ in range(hidden_layers):
            layers.append(Block(hidden_size))
        layers.append(nn.LayerNorm(hidden_size))
        layers.append(nn.Linear(hidden_size, self.data_size))

        self.joint_mlp = nn.Sequential(*layers)

    def forward(self, x, t):
        t_emb = self.time_emb(t.reshape(-1,1))
        x = torch.cat((x, t_emb), dim=-1)
        return self.joint_mlp(x)

class Diffusion():
    def __init__(self,
                 num_timesteps=1000,
                 beta_start=0.0001,
                 beta_end=0.02,
                 device="cpu"):  

        self.device = device
        self.num_timesteps = num_timesteps

        self.betas = torch.linspace(
            beta_start, beta_end, num_timesteps, dtype=torch.float32, device=device)

        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, axis=0)
        self.alphas_cumprod_prev = F.pad(
            self.alphas_cumprod[:-1], (1, 0), value=1.)

        self.sqrt_alphas_cumprod = self.alphas_cumprod ** 0.5
        self.sqrt_one_minus_alphas_cumprod = (1 - self.alphas_cumprod) ** 0.5

        self.sqrt_inv_alphas = torch.sqrt(1. / self.alphas)
        self.noise_coef = self.betas / self.sqrt_one_minus_alphas_cumprod
        self.variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)

    def add_noise(self, x_start, x_noise, timesteps):
        # compute x_t for DDPM algorithm 1
        s1 = self.sqrt_alphas_cumprod[timesteps].reshape(-1, 1)
        s2 = self.sqrt_one_minus_alphas_cumprod[timesteps].reshape(-1, 1)
        return s1 * x_start + s2 * x_noise
    
    def sample_step(self, model_output, timestep, sample):
        # compute x_{t-1} for DDPM algorithm 2
        s1 = self.sqrt_inv_alphas[timestep].reshape(-1, 1)
        s2 = self.noise_coef[timestep].reshape(-1, 1)
        s3 = self.variance[timestep].reshape(-1, 1) ** 0.5
        noise = torch.randn_like(model_output)
        return s1 * (sample - s2 * model_output) + s3 * noise

    def __len__(self):
        return self.num_timesteps



In [ ]:
experiment_name = "base"
dataset = "pose"
train_batch_size = 256
eval_batch_size= 1000
num_epochs= 100
learning_rate= 1e-3
num_timesteps= 200
embedding_size= 32
hidden_size= 32
hidden_layers= 1
eval_path= None
device= "cuda"

outdir = f"exps/{experiment_name}"

In [ ]:

print(device)
dataset = get_dataset(dataset)
dataloader = DataLoader(
    dataset, batch_size=train_batch_size, shuffle=True, drop_last=True)

model = MLP(
    data_dim=x0.shape[1],
    hidden_size=hidden_size,
    hidden_layers=hidden_layers,
    emb_size=embedding_size).to(device)

diffusion = Diffusion(
    num_timesteps=num_timesteps,
    device=device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: 1 - step / (num_epochs * len(dataloader)))

In [ ]:
if eval_path is None:
    global_step = 0
    losses = []
    print("Training model...")
    for epoch in range(num_epochs):
        model.train()
        progress_bar = tqdm(total=len(dataloader))
        progress_bar.set_description(f"Epoch {epoch}")
        for step, batch in enumerate(dataloader):
            if type(batch) == list:
                batch = batch[0]
            batch = batch.to(device)

            noise = torch.randn_like(batch).to(device)  
            timesteps = torch.randint(
                0, diffusion.num_timesteps, (batch.shape[0],),
                device=device  
            ).long()

            noisy = diffusion.add_noise(batch, noise, timesteps)
            noise_pred = model(noisy.to(device), timesteps.to(device))
            loss = F.mse_loss(noise_pred, noise.to(device))
            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

            progress_bar.update(1)
            logs = {"loss": loss.detach().item(), "step": global_step, "lr": scheduler.get_last_lr()[0]}
            losses.append(loss.detach().item())
            progress_bar.set_postfix(**logs)
            global_step += 1
        progress_bar.close()

    print("Saving model...")
    os.makedirs(outdir, exist_ok=True)
    torch.save(model.state_dict(), f"{outdir}/model.pt")